Nama : Alif Ilham Permata

NIM : 2411511012

Jurusan : Teknik Komputer B


1. **Dining Philosophers dengan N Berbeda:** Ulangi solusi arbitrator (bagian 1) dengan N = 3, 7, dan 10 filsuf. Ukur throughput (total makan per detik) untuk masing-masing N. Apakah throughput naik linear terhadap N? Jelaskan mengapa (atau mengapa tidak).

In [1]:
import threading
import time

def jalankan_simulasi_dining(N, durasi_detik=2.0):
    forks = [threading.Lock() for _ in range(N)]
    kursi_meja = threading.Semaphore(N - 1)
    selesai_makan = [0] * N
    berhenti = threading.Event()

    def filsuf(i):
        kiri = forks[i]
        kanan = forks[(i + 1) % N]
        while not berhenti.is_set():
            with kursi_meja:
                kiri.acquire()
                kanan.acquire()
                selesai_makan[i] += 1
                time.sleep(0.001)  # Simulasi makan singkat untuk throughput tinggi
                kanan.release()
                kiri.release()
            time.sleep(0.001)  # Simulasi berpikir singkat

    threads = [threading.Thread(target=filsuf, args=(i,)) for i in range(N)]
    t_start = time.perf_counter()
    for t in threads:
        t.start()

    time.sleep(durasi_detik)
    berhenti.set()

    for t in threads:
        t.join()
    t_end = time.perf_counter()

    total_makan = sum(selesai_makan)
    durasi_aktual = t_end - t_start
    throughput = total_makan / durasi_aktual
    return total_makan, throughput, selesai_makan

for n_filsuf in [3, 7, 10]:
    total, tp, detail = jalankan_simulasi_dining(n_filsuf)
    print(f"N = {n_filsuf} -> Total Makan: {total}, Throughput: {tp:.2f} makan/detik")
    print(f"  Detail makan per filsuf: {detail}\n")

N = 3 -> Total Makan: 1766, Throughput: 880.96 makan/detik
  Detail makan per filsuf: [588, 589, 589]

N = 7 -> Total Makan: 4008, Throughput: 1997.00 makan/detik
  Detail makan per filsuf: [575, 573, 573, 574, 570, 571, 572]

N = 10 -> Total Makan: 4300, Throughput: 2138.22 makan/detik
  Detail makan per filsuf: [431, 430, 430, 429, 428, 428, 430, 431, 430, 433]



**Analisis Throughput terhadap N:**
Throughput **tidak naik secara linear sempurna** terhadap $N$.

Hal ini disebabkan oleh:
1. **Overhead Sinkronisasi & Context Switching:** Semakin banyak thread ($N$ bertambah), overhead bagi scheduler OS untuk melakukan context switching antar thread semakin besar.
2. **Arbitrator Bottleneck:** Semaphore pembatas `kursi_meja` berukuran $N-1$ bertindak sebagai lock global tunggal di mana semua thread harus bersaing memperebutkannya. Kompetisi perebutan lock (lock contention) ini membatasi efisiensi paralelisme maksimal (Hukum Amdahl).

2. **Readers-Writers dengan Prioritas Writer:** Modifikasi solusi fair (bagian 2) menjadi solusi yang secara eksplisit **memprioritaskan writer** — begitu ada writer menunggu, TIDAK ADA reader baru yang boleh mulai membaca sampai writer tersebut selesai (reader yang sudah membaca boleh menyelesaikan bacaannya). Uji apakah reader baru sekarang yang berisiko starvation jika writer datang terus-menerus.

In [2]:
import threading
import time

read_count = 0
write_count = 0

mutex_read = threading.Lock()   # Melindungi read_count
mutex_write = threading.Lock()  # Melindungi write_count

read_lock = threading.Lock()    # Mengunci masuknya reader
write_lock = threading.Lock()   # Mengunci resource untuk penulisan eksklusif

data = 0
log_rw = []

def reader_prioritas_writer(id_r):
    global read_count
    # Reader harus memeriksa apakah ada writer yang antre/menulis via read_lock
    read_lock.acquire()
    with mutex_read:
        read_count += 1
        if read_count == 1:
            write_lock.acquire() # Reader pertama mengunci akses writer
    read_lock.release()

    # Proses membaca
    log_rw.append(f"[Reader-{id_r}] membaca data: {data}")
    time.sleep(0.05)

    with mutex_read:
        read_count -= 1
        if read_count == 0:
            write_lock.release() # Reader terakhir melepas kunci untuk writer

def writer_prioritas_writer(id_w):
    global write_count, data
    with mutex_write:
        write_count += 1
        if write_count == 1:
            read_lock.acquire() # Writer pertama menutup pintu bagi reader baru

    with write_lock:
        data += 1
        log_rw.append(f"[Writer-{id_w}] MENULIS data baru: {data}")
        time.sleep(0.1)

    with mutex_write:
        write_count -= 1
        if write_count == 0:
            read_lock.release() # Writer terakhir membuka kembali pintu untuk reader

# Simulasi pembuktian starvation bagi reader
threads = []
# Jalankan beberapa reader mula-mula
for i in range(2):
    threads.append(threading.Thread(target=reader_prioritas_writer, args=(i,)))

# Jalankan writer terus-menerus
for i in range(3):
    threads.append(threading.Thread(target=writer_prioritas_writer, args=(i,)))

# Jalankan reader baru yang datang belakangan
for i in range(2, 5):
    threads.append(threading.Thread(target=reader_prioritas_writer, args=(i,)))

for t in threads:
    t.start()
for t in threads:
    t.join()

for log in log_rw:
    print(log)

[Reader-0] membaca data: 0
[Reader-1] membaca data: 0
[Writer-0] MENULIS data baru: 1
[Writer-1] MENULIS data baru: 2
[Writer-2] MENULIS data baru: 3
[Reader-2] membaca data: 3
[Reader-3] membaca data: 3
[Reader-4] membaca data: 3


3. **Sleeping Barber dengan Banyak Tukang Cukur:** Modifikasi bagian 4 menjadi **2 tukang cukur** yang berbagi satu ruang tunggu yang sama. Pikirkan bagaimana menentukan tukang cukur mana yang melayani pelanggan berikutnya (hint: gunakan `Semaphore` tambahan atau `Queue` untuk mengelola giliran antar-tukang-cukur).

In [3]:
import threading
import time
import random
import queue

KURSI_TUNGGU = 3
log_lock = threading.Lock()

def catat(msg):
    with log_lock:
        print(msg)

# Menggunakan thread-safe Queue untuk mengelola koordinasi pelanggan
antrean_kursi = queue.Queue(maxsize=KURSI_TUNGGU)
sinyal_potong = {}

def tukang_cukur(id_barber):
    while True:
        catat(f"[Tukang Cukur-{id_barber}] tidur menunggu pelanggan...")
        # Mengambil pelanggan dari antrean (akan block otomatis jika kosong)
        id_pelanggan = antrean_kursi.get()

        catat(f"[Tukang Cukur-{id_barber}] mulai mencukur Pelanggan-{id_pelanggan}...")
        time.sleep(0.15) # Simulasi potong rambut
        catat(f"[Tukang Cukur-{id_barber}] selesai mencukur Pelanggan-{id_pelanggan}")

        # Sinyalkan kepada pelanggan tersebut bahwa pencukuran selesai
        sinyal_potong[id_pelanggan].release()
        antrean_kursi.task_done()

def pelanggan(id_pelanggan):
    # Inisialisasi semaphore personal untuk menunggu giliran potong
    sinyal_potong[id_pelanggan] = threading.Semaphore(0)
    try:
        # Mencoba duduk di kursi tunggu tanpa block jika penuh
        antrean_kursi.put_nowait(id_pelanggan)
        catat(f"  [Pelanggan-{id_pelanggan}] berhasil duduk di ruang tunggu ({antrean_kursi.qsize()}/{KURSI_TUNGGU})")

        # Menunggu sampai selesai dicukur oleh salah satu barber
        sinyal_potong[id_pelanggan].acquire()
    except queue.Full:
        catat(f"  [Pelanggan-{id_pelanggan}] melihat ruang tunggu PENUH, lalu pergi!")

# Jalankan 2 barber daemon
for b_id in range(1, 3):
    threading.Thread(target=tukang_cukur, args=(b_id,), daemon=True).start()

random.seed(42)
pelanggan_threads = []
for i in range(10):
    time.sleep(random.uniform(0.01, 0.08))
    t = threading.Thread(target=pelanggan, args=(i,))
    t.start()
    pelanggan_threads.append(t)

for t in pelanggan_threads:
    t.join()

time.sleep(0.5) # Berikan jeda sejenak agar barber kembali tidur sebelum program selesai
print("\nSemua pelanggan selesai diproses.")

[Tukang Cukur-1] tidur menunggu pelanggan...
[Tukang Cukur-2] tidur menunggu pelanggan...
  [Pelanggan-0] berhasil duduk di ruang tunggu (1/3)
[Tukang Cukur-1] mulai mencukur Pelanggan-0...
  [Pelanggan-1] berhasil duduk di ruang tunggu (1/3)
[Tukang Cukur-2] mulai mencukur Pelanggan-1...
  [Pelanggan-2] berhasil duduk di ruang tunggu (1/3)
  [Pelanggan-3] berhasil duduk di ruang tunggu (2/3)
  [Pelanggan-4] berhasil duduk di ruang tunggu (3/3)
[Tukang Cukur-1] selesai mencukur Pelanggan-0
[Tukang Cukur-1] tidur menunggu pelanggan...
[Tukang Cukur-1] mulai mencukur Pelanggan-2...
[Tukang Cukur-2] selesai mencukur Pelanggan-1
[Tukang Cukur-2] tidur menunggu pelanggan...
[Tukang Cukur-2] mulai mencukur Pelanggan-3...
  [Pelanggan-5] berhasil duduk di ruang tunggu (2/3)
  [Pelanggan-6] berhasil duduk di ruang tunggu (3/3)
  [Pelanggan-7] melihat ruang tunggu PENUH, lalu pergi!
[Tukang Cukur-1] selesai mencukur Pelanggan-2
[Tukang Cukur-1] tidur menunggu pelanggan...
[Tukang Cukur-1] mulai

4. Refleksi singkat (maks. 150 kata): Dari kelima classical problem yang dibahas (Dining Philosophers, Readers-Writers, Producer-Consumer, Sleeping Barber, Cigarette Smokers), menurut kalian mana yang paling mirip dengan masalah nyata yang mungkin kalian temui dalam pengembangan software (web server, database, sistem embedded, dsb)? Berikan contoh konkret.

Dari kelima masalah klasik sinkronisasi, **Producer-Consumer Problem** adalah yang paling sering ditemui dalam pengembangan software dunia nyata.

Contoh konkretnya adalah arsitektur **Message Broker / Task Queue** (seperti Celery, RabbitMQ, atau Redis) pada web server skala besar. Ketika user mengunggah foto profil, server web bertindak sebagai *Producer* yang meletakkan tugas pemrosesan gambar (resize, kompresi) ke dalam antrean (Queue). Server worker di latar belakang bertindak sebagai *Consumer* yang mengambil tugas tersebut dan memprosesnya secara asinkron tanpa mengganggu responsivitas antarmuka pengguna utama. Pola ini sangat krusial untuk menjaga stabilitas performa sistem web modern.